In [ ]:
import os
from pathlib import Path
import pandas as pd
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib.pyplot as plt


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
# https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html
base_ds = datasets.ImageFolder(root=DATA_DIR)
base_ds

In [ ]:
folder_index = np.array([index for _, index in base_ds.samples])

In [ ]:
data_stats = []
for flow_type in os.listdir(DATA_DIR):
    num_images = len(os.listdir(DATA_DIR / flow_type))
    data_stats.append({"Flow Type": flow_type, "Image Count": num_images})

data_stats = pd.DataFrame(data_stats)
data_stats

In [ ]:
rng = 42

# Split data into train and test/val sets
# https://scikit-learn.org/stable/modules/cross_validation.html#stratified-shuffle-split
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html

# In an image each pixels RGB values is a feature, for the sake of splitting in 
# this classification problem we only need to consider the classification of the image itself first
dummy_features = np.zeros_like(folder_index)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=rng)
train_index, test_and_val_index = next(sss1.split(dummy_features, folder_index))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=rng)
test_and_val_lables = folder_index[test_and_val_index]
dummy_features = np.zeros_like(test_and_val_lables)
test_index_rel, val_index_rel = next(sss2.split(dummy_features, test_and_val_lables))

# Second SSS splits relative to the index of test_and_val lables so these need
# to be mapped back to original indices
test_index = test_and_val_index[test_index_rel]
val_index  = test_and_val_index[val_index_rel]

In [ ]:
class_names = list(base_ds.class_to_idx.keys())
classes = np.arange(len(class_names))

train_counts = [np.sum(folder_index[train_index] == i) for i in classes]
val_counts   = [np.sum(folder_index[val_index] == i) for i in classes]
test_counts  = [np.sum(folder_index[test_index] == i) for i in classes]

fig, ax = plt.subplots()
bar_width = 0.6
ax.bar(class_names, train_counts, label='Train', color='tab:blue', width=bar_width)
ax.bar(class_names, val_counts, bottom=train_counts, label='Validation', color='tab:orange', width=bar_width)
ax.bar(class_names, test_counts, bottom=np.array(train_counts)+np.array(val_counts), 
       label='Test', color='tab:green', width=bar_width)

ax.set_xlabel('Folder / Class')
ax.set_ylabel('Number of Images')
ax.set_title('Dataset Split per Class')
ax.legend()

In [ ]:
# Image preprocessing and augmentation
image_size = 224
train_tf = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf  = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

In [ ]:
# Establish subsets
train_ds = Subset(datasets.ImageFolder(root=DATA_DIR, transform=train_tf), train_index)
val_ds   = Subset(datasets.ImageFolder(root=DATA_DIR, transform=eval_tf),  val_index)
test_ds  = Subset(datasets.ImageFolder(root=DATA_DIR, transform=eval_tf),  test_index)